# Chapter 8 — Putting It All Together: Forward, Backward, AdamW

> Course: **llm.c — Zero to Hero**, Chapter 8 of ~20.  **Milestone chapter for Part I.**
> Builds on Chapters 2–7 (every layer's forward and backward pass).

You've now written **every layer** in `train_gpt2.c` — encoder, layernorm, matmul, attention, GELU, residual, softmax, cross-entropy. Time to see how they snap together into a real GPT-2 training loop.

This chapter is mostly **orchestration walkthrough**: the architectural patterns that turn 8 layer functions into a 124M-parameter model. The two runnable pieces are the **unified-malloc pattern** (one giant `malloc` that backs every parameter tensor) and the **AdamW optimizer step** (verified against `torch.optim.AdamW`).

By the end of Part I, you'll be able to read every line of `train_gpt2.c` and explain it. Part II will then pivot to GPU.

### Learning objectives

By the end of this chapter you will:

- Read the **`ParameterTensors`** and **`ActivationTensors`** structs and explain why they exist.
- Implement the **single-`malloc` + struct-of-pointers** idiom yourself (every production ML framework uses some version of this).
- Trace `gpt2_forward` from `inputs` to `losses`, naming every layer call.
- Trace `gpt2_backward` from `dlosses` back to `dwte` / `dwpe`, in reverse.
- Derive AdamW from scratch and code one update step in C, matching `torch.optim.AdamW` numerically.


## 1. Concept — How Do You Carry 124M Parameters in C?

GPT-2 small has 124 million parameters spread across 16 distinct tensors (one per kind of weight) **stacked across L=12 layers** (so the per-layer ones become `(L, ...)` tensors). Some are shape `(50304, 768)` (the embedding); others are tiny like `(768,)` (the final-LN bias).

PyTorch hides this with `nn.Module`:

```python
class GPT2(nn.Module):
    def __init__(self, ...):
        self.wte = nn.Embedding(V, C)
        self.h = nn.ModuleList([Block(...) for _ in range(L)])
        self.lnf = nn.LayerNorm(C)
```

In C there's no `nn.Module`. So `llm.c` does the equivalent by hand, using **two clever idioms**:

1. **The `ParameterTensors` struct of pointers** — 16 named `float*` fields, one per logical tensor. Calling code reads `params.qkvw` like a typed handle.
2. **One giant `malloc` for the whole model** — a single contiguous buffer holds *all* weights end-to-end. The struct fields are just pointers into different *offsets* of that buffer.

This pairing is the single most important architectural pattern in `train_gpt2.c`. Once you see it, every memory operation in the codebase makes sense.


## 2. Walkthrough — `ParameterTensors`

From [`train_gpt2.c`](train_gpt2.c) lines 535–554:

```c
#define NUM_PARAMETER_TENSORS 16
typedef struct {
    float* wte;       // (V, C)
    float* wpe;       // (maxT, C)
    float* ln1w;      // (L, C)        per-layer LN1 weight
    float* ln1b;      // (L, C)        per-layer LN1 bias
    float* qkvw;      // (L, 3*C, C)   per-layer QKV projection weight
    float* qkvb;      // (L, 3*C)      per-layer QKV projection bias
    float* attprojw;  // (L, C, C)     per-layer attention output projection
    float* attprojb;  // (L, C)
    float* ln2w;      // (L, C)        per-layer LN2 weight
    float* ln2b;      // (L, C)
    float* fcw;       // (L, 4*C, C)   per-layer FFN up-projection
    float* fcb;       // (L, 4*C)
    float* fcprojw;   // (L, C, 4*C)   per-layer FFN down-projection
    float* fcprojb;   // (L, C)
    float* lnfw;      // (C)           final LN weight
    float* lnfb;      // (C)           final LN bias
} ParameterTensors;
```

Three things to notice:

1. **The `(L, ...)` prefix** on most tensors. `ln1w` isn't shape `(C,)` — it's shape `(L, C)`. All 12 layers' LN1 weights are stacked into one `(L, C)` block, and at runtime `gpt2_forward` does `params.ln1w + l*C` to grab layer `l`'s slice.
2. **`wte` is reused as the unembedding head.** `gpt2_forward` calls `matmul_forward(acts.logits, acts.lnf, params.wte, NULL, B, T, C, Vp);` — passing the *same* `wte` weight for both the embedding lookup at the input and the logits projection at the output. This is **weight tying** — saves ~40M parameters (V·C is the biggest tensor in the model).
3. **The struct is exactly `NUM_PARAMETER_TENSORS = 16` fields.** That number is hardcoded so the iteration code can use a fixed-size array of `float**`.


## 3. Walkthrough — `malloc_and_point_parameters`

The unified-malloc pattern in 20 lines, from [`train_gpt2.c`](train_gpt2.c) lines 580–599:

```c
float* malloc_and_point_parameters(ParameterTensors* params, size_t* param_sizes) {
    size_t num_parameters = 0;
    for (size_t i = 0; i < NUM_PARAMETER_TENSORS; i++) {
        num_parameters += param_sizes[i];
    }
    // ONE big malloc for all 124M parameters
    float* params_memory = (float*) mallocCheck(num_parameters * sizeof(float));

    // build an array of (struct field address) pointers — one per tensor
    float** ptrs[] = {
        &params->wte, &params->wpe, &params->ln1w, &params->ln1b,
        &params->qkvw, &params->qkvb, &params->attprojw, &params->attprojb,
        &params->ln2w, &params->ln2b, &params->fcw, &params->fcb,
        &params->fcprojw, &params->fcprojb, &params->lnfw, &params->lnfb
    };

    // walk through the buffer assigning each tensor a slice
    float* iter = params_memory;
    for (size_t i = 0; i < NUM_PARAMETER_TENSORS; i++) {
        *(ptrs[i]) = iter;     // params->wte = iter; (then params->wpe = iter; etc.)
        iter += param_sizes[i];
    }
    return params_memory;
}
```

Why bother? **One contiguous buffer is dramatically friendlier than 16 separate `malloc`s**:

- **Loading weights is one `fread`** — read `num_parameters * 4` bytes from disk straight into `params_memory`, and all 16 tensors are populated. Compare to 16 separate reads with their own bookkeeping.
- **Optimizer state mirrors it perfectly.** `m_memory` and `v_memory` (AdamW's two moments) are also single `(num_parameters,)` arrays. The optimizer can do `for (i = 0; i < num_parameters; i++)` over all parameters at once without caring about which tensor is which.
- **`grads_memory` is a clone with the same layout.** Notice in the `GPT2` struct:
  ```c
  ParameterTensors grads;
  float* grads_memory;
  ```
  We have a *second* `ParameterTensors` whose pointers index into `grads_memory` at the *same offsets* as `params` does into `params_memory`. So `grads.wte[i]` is the gradient of `params.wte[i]`, automatically.

This is the **structure-of-arrays pattern** taken to its logical end. Calling code gets typed access (`params.qkvw`); under the hood it's all one block of floats.


## 4. Walkthrough — `ActivationTensors`

From [`train_gpt2.c`](train_gpt2.c) lines 601–626. There are **23 activation tensors** because every layer that needs to cache something for backward gets its own buffer:

```c
typedef struct {
    float* encoded;     // (B, T, C)
    float* ln1;         // (L, B, T, C)       LN1 outputs, all layers
    float* ln1_mean;    // (L, B, T)          LN1 mean cache
    float* ln1_rstd;    // (L, B, T)          LN1 rstd cache  ← Chapter 3 trick
    float* qkv;         // (L, B, T, 3*C)     post-QKV projection
    float* atty;        // (L, B, T, C)       post-attention
    float* preatt;      // (L, B, NH, T, T)   pre-softmax attention scores
    float* att;         // (L, B, NH, T, T)   post-softmax attention weights ← Chapter 6 cache
    float* attproj;     // (L, B, T, C)       post-attention-output-projection
    float* residual2;   // (L, B, T, C)       after first residual add
    float* ln2;         // (L, B, T, C)       LN2 outputs
    float* ln2_mean;    // (L, B, T)
    float* ln2_rstd;    // (L, B, T)
    float* fch;         // (L, B, T, 4*C)     pre-GELU FFN hidden
    float* fch_gelu;    // (L, B, T, 4*C)     post-GELU FFN hidden
    float* fcproj;      // (L, B, T, C)       FFN output
    float* residual3;   // (L, B, T, C)       after second residual add
    float* lnf;         // (B, T, C)          final LN
    float* lnf_mean;    // (B, T)
    float* lnf_rstd;    // (B, T)
    float* logits;      // (B, T, Vp)
    float* probs;       // (B, T, Vp)         ← Chapter 7 cache
    float* losses;      // (B, T)
} ActivationTensors;
```

Notice: every `*_mean`, `*_rstd`, `att`, and `probs` you see is one of the **caching tricks** we discussed in earlier chapters. The forward saves them; the backward reads them.

### Lazy allocation

Look at the top of `gpt2_forward`:

```c
if (model->acts_memory == NULL) {
    fill_in_activation_sizes(model->act_sizes, model->config, B, T);
    ...
    model->acts_memory = malloc_and_point_activations(&model->acts, model->act_sizes);
}
```

Activation sizes depend on **runtime** `B` and `T` (whereas parameter sizes only depend on the static config). So `llm.c` doesn't allocate activations until the first `gpt2_forward` call — at which point `B, T` are known. The allocation is then **frozen** for the model's lifetime; subsequent forwards must use the same `B, T`.

Same idiom: 23 fields, one giant `malloc`, struct of pointers.


## 5. Walkthrough — `gpt2_forward`

From [`train_gpt2.c`](train_gpt2.c) lines 821–891. The orchestration **after** the lazy allocation is just calling each layer in order:

```c
encoder_forward(acts.encoded, inputs, params.wte, params.wpe, B, T, C);

for (int l = 0; l < L; l++) {
    residual = (l == 0) ? acts.encoded
                        : acts.residual3 + (l-1)*B*T*C;

    // pull per-layer pointer slices (params.ln1w + l*C, etc.)
    float* l_ln1w = params.ln1w + l*C;
    /* … 11 more … */

    layernorm_forward(l_ln1, l_ln1_mean, l_ln1_rstd, residual, l_ln1w, l_ln1b, B, T, C);
    matmul_forward    (l_qkv, l_ln1, l_qkvw, l_qkvb, B, T, C, 3*C);
    attention_forward (l_atty, l_preatt, l_att, l_qkv, B, T, C, NH);
    matmul_forward    (l_attproj, l_atty, l_attprojw, l_attprojb, B, T, C, C);
    residual_forward  (l_residual2, residual, l_attproj, B*T*C);
    layernorm_forward (l_ln2, l_ln2_mean, l_ln2_rstd, l_residual2, l_ln2w, l_ln2b, B, T, C);
    matmul_forward    (l_fch, l_ln2, l_fcw, l_fcb, B, T, C, 4*C);
    gelu_forward      (l_fch_gelu, l_fch, B*T*4*C);
    matmul_forward    (l_fcproj, l_fch_gelu, l_fcprojw, l_fcprojb, B, T, 4*C, C);
    residual_forward  (l_residual3, l_residual2, l_fcproj, B*T*C);
}
residual = acts.residual3 + (L-1)*B*T*C;
layernorm_forward(acts.lnf, acts.lnf_mean, acts.lnf_rstd, residual, params.lnfw, params.lnfb, B, T, C);
matmul_forward   (acts.logits, acts.lnf, params.wte, NULL, B, T, C, Vp);   // ← weight tying: reuses params.wte!
softmax_forward  (acts.probs, acts.logits, B, T, V, Vp);
crossentropy_forward(acts.losses, acts.probs, targets, B, T, Vp);
```

That's the whole forward. Each line is a function you've already written in Chapters 2–7.

### The Transformer block, in 10 calls

Per layer:

1. `layernorm_forward` — LN1, the pre-attention norm
2. `matmul_forward`   — QKV projection (one matmul, output is `(B,T,3C)`)
3. `attention_forward` — multi-head causal self-attention
4. `matmul_forward`   — output projection (`(C,C)`)
5. `residual_forward` — first residual add
6. `layernorm_forward` — LN2, the pre-FFN norm
7. `matmul_forward`   — FFN up-projection (`C → 4C`)
8. `gelu_forward`     — element-wise GELU
9. `matmul_forward`   — FFN down-projection (`4C → C`)
10. `residual_forward` — second residual add

That's a **single Transformer block**. GPT-2 small has 12 of them. The function calls form an exact 1-to-1 mapping with the architecture diagram you'd find in any "GPT-2 explained" blog post.


## 6. Walkthrough — `gpt2_backward`

The backward, from [`train_gpt2.c`](train_gpt2.c) lines 898–1005, is the forward in reverse:

```c
// kick off chain rule with dL/dloss = 1/(B*T) for each loss term
float dloss_mean = 1.0f / (B*T);
for (int i = 0; i < B*T; i++) grads_acts.losses[i] = dloss_mean;

crossentropy_softmax_backward(grads_acts.logits, grads_acts.losses, acts.probs, model->targets, B, T, V, Vp);
matmul_backward (grads_acts.lnf, grads.wte, NULL, grads_acts.logits, acts.lnf, params.wte, B, T, C, Vp);
layernorm_backward(dresidual, grads.lnfw, grads.lnfb, grads_acts.lnf, residual, ..., B, T, C);

for (int l = L-1; l >= 0; l--) {                          // <-- REVERSE iteration
    /* … same per-layer pointer setup … */
    residual_backward (dl_residual2, dl_fcproj, dl_residual3, B*T*C);
    matmul_backward   (dl_fch_gelu, dl_fcprojw, dl_fcprojb, dl_fcproj, l_fch_gelu, l_fcprojw, B, T, 4*C, C);
    gelu_backward     (dl_fch, l_fch, dl_fch_gelu, B*T*4*C);
    matmul_backward   (dl_ln2, dl_fcw, dl_fcb, dl_fch, l_ln2, l_fcw, B, T, C, 4*C);
    layernorm_backward(dl_residual2, dl_ln2w, dl_ln2b, dl_ln2, l_residual2, ..., B, T, C);
    residual_backward (dresidual, dl_attproj, dl_residual2, B*T*C);
    matmul_backward   (dl_atty, dl_attprojw, dl_attprojb, dl_attproj, l_atty, l_attprojw, B, T, C, C);
    attention_backward(dl_qkv, dl_preatt, dl_att, dl_atty, l_qkv, l_att, B, T, C, NH);
    matmul_backward   (dl_ln1, dl_qkvw, dl_qkvb, dl_qkv, l_ln1, l_qkvw, B, T, C, 3*C);
    layernorm_backward(dresidual, dl_ln1w, dl_ln1b, dl_ln1, residual, ..., B, T, C);
}
encoder_backward(grads.wte, grads.wpe, grads_acts.encoded, model->inputs, B, T, C);
```

### Three essential tricks

1. **Initialize `dlosses` with `1/(B*T)`.** That's the gradient of `mean(losses)` with respect to one entry of `losses`. From there, every `*_backward` call propagates upstream gradients to the weight gradients (`grads.*`) and the activation gradients (`grads_acts.*`).

2. **`+=` for `dinp`** in every layer's backward (Chapters 3–6). When backprop runs for layer `l`, the same `dl_residual2` (or `dresidual`) buffer is touched both by the FFN backward path *and* by the attention backward path through the residual connection. The first call writes; the second call adds. By the end of the loop iteration, `dresidual` holds the **sum** of gradients flowing through both paths — which is exactly the chain rule for `out = inp + branch(inp)`.

3. **Weight tying ↔ shared `grads.wte`.** Look at the line near the top:
   ```c
   matmul_backward(grads_acts.lnf, grads.wte, NULL, ...)
   ```
   That writes the gradient of the *unembedding* into `grads.wte` (the same tensor used as the embedding lookup at the input). Then at the very end:
   ```c
   encoder_backward(grads.wte, grads.wpe, ..., B, T, C);
   ```
   ...the embedding backward **`+=` accumulates** into the *same* `grads.wte`. Both gradient paths land on the same buffer, automatically combined. This is why weight tying "just works" — the unified-malloc + `+=` pattern handles it for free.


## 7. `gpt2_zero_grad` and Lazy Buffer Allocation

The simplest function in the file ([line 893](train_gpt2.c#L893)):

```c
void gpt2_zero_grad(GPT2 *model) {
    if (model->grads_memory != NULL)      memset(model->grads_memory,      0, model->num_parameters * sizeof(float));
    if (model->grads_acts_memory != NULL) memset(model->grads_acts_memory, 0, model->num_activations * sizeof(float));
}
```

Two `memset(0)` calls and you're done. Why is it that simple? Because all the gradient buffers are **single contiguous blocks** (Section 3). Zeroing every gradient = zeroing one big region of memory.

Compare what PyTorch does: `optimizer.zero_grad()` walks every `param.grad` tensor in a Python loop and zeroes each. Many small ops vs one big `memset`. The C version wins both in code clarity and in raw speed (one big `memset` is trivially vectorized by the C library).

Note also the **first-time allocation** dance in `gpt2_backward`:

```c
if (model->grads_memory == NULL) {
    model->grads_memory = malloc_and_point_parameters(&model->grads, model->param_sizes);
    model->grads_acts_memory = malloc_and_point_activations(&model->grads_acts, model->act_sizes);
    gpt2_zero_grad(model);
}
```

Same lazy pattern as activations: don't allocate gradient buffers until the first `backward()`. Inference-only callers (no `targets`) never pay for them.


## 8. AdamW — The Math

AdamW is Adam with **decoupled weight decay**. Per parameter $\theta_i$ at step $t$:

1. **Gradient**: $g_t$ (already in `grads_memory[i]` after backward).
2. **First moment** (smoothed gradient, "momentum"):
$$m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t$$
3. **Second moment** (smoothed squared gradient, "RMSprop"):
$$v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2$$
4. **Bias correction** — early steps have $m_0 = v_0 = 0$, so the smoothed values underestimate the truth. Correct for it:
$$\hat m_t = \frac{m_t}{1 - \beta_1^t} \qquad \hat v_t = \frac{v_t}{1 - \beta_2^t}$$
5. **Update** with decoupled weight decay:
$$\theta_t = \theta_{t-1} - \alpha \left( \frac{\hat m_t}{\sqrt{\hat v_t} + \varepsilon} + \lambda\, \theta_{t-1} \right)$$

Notation: $\alpha$ = learning rate, $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\varepsilon = 10^{-8}$, $\lambda$ = weight decay (often $0.01$).

Three per-parameter pieces of state: the parameter $\theta$ itself, plus $m$ and $v$. **For a 124M-parameter model, that's 3 × 124M = 372M floats = 1.5 GB** of optimizer state. AdamW is famously memory-hungry — and you'll feel why when we get to ZeRO sharding in Chapter 20.

### Why "decoupled" weight decay?

Original Adam adds $\lambda \theta$ to the gradient *before* the moment update. AdamW adds it **directly to the parameter** at update time. The difference matters: in Adam, weight decay gets dampened by the $1/\sqrt{\hat v}$ adaptive scaling (small-gradient parameters get large effective decay, which is wrong). In AdamW, weight decay is uniform across parameters. Loshchilov & Hutter showed this generalizes better, and every modern Transformer trainer uses AdamW.


## 9. The C `gpt2_update`

From [`train_gpt2.c`](train_gpt2.c) lines 1007–1033. This is **AdamW in 18 lines**:

```c
void gpt2_update(GPT2 *model, float learning_rate, float beta1, float beta2,
                 float eps, float weight_decay, int t) {
    if (model->m_memory == NULL) {
        model->m_memory = (float*) calloc(model->num_parameters, sizeof(float));
        model->v_memory = (float*) calloc(model->num_parameters, sizeof(float));
    }

    for (size_t i = 0; i < model->num_parameters; i++) {
        float param = model->params_memory[i];
        float grad  = model->grads_memory[i];

        float m = beta1 * model->m_memory[i] + (1.0f - beta1) * grad;
        float v = beta2 * model->v_memory[i] + (1.0f - beta2) * grad * grad;
        float m_hat = m / (1.0f - powf(beta1, t));
        float v_hat = v / (1.0f - powf(beta2, t));

        model->m_memory[i] = m;
        model->v_memory[i] = v;
        model->params_memory[i] -= learning_rate * (m_hat / (sqrtf(v_hat) + eps)
                                                  + weight_decay * param);
    }
}
```

Three things to notice:

1. **One `for` loop over `num_parameters`.** No nested layer loops, no struct field iteration. This is only possible because of the unified malloc — `params_memory[i]`, `grads_memory[i]`, `m_memory[i]`, `v_memory[i]` *all line up* at index `i`, no matter which logical tensor `i` belongs to.
2. **Lazy allocation of optimizer state** on the first call. Same pattern as gradients.
3. **`t` is the step number (1-indexed)**. The bias-correction terms `1 - β^t` shrink toward 0 over training, but only matter much for `t < ~100`.

This loop runs over all 124 million parameters every step. It's one of the simplest hot loops in the codebase, but at 124M iterations per step × every training step, it adds up.


## 10. Demo 1 — Build the Unified-Malloc Pattern Yourself

Let's actually build a tiny `ParameterTensors` that demonstrates the pattern with three named tensors of different sizes. You'll see how the struct fields point into one contiguous block.


In [ ]:
!mkdir -p course/ch08_build


In [ ]:
%%writefile course/ch08_build/unified_malloc.c
#include <stdio.h>
#include <stdlib.h>

#define NUM_TENSORS 3

// A tiny ParameterTensors with 3 named buffers
typedef struct {
    float* a;   // size 5
    float* b;   // size 3
    float* c;   // size 4
} ToyParams;

float* malloc_and_point(ToyParams* p, size_t* sizes) {
    size_t total = 0;
    for (int i = 0; i < NUM_TENSORS; i++) total += sizes[i];

    float* mem = (float*) malloc(total * sizeof(float));
    if (!mem) { perror("malloc"); exit(1); }

    // The classic llm.c pattern: array of (struct field address) pointers
    float** ptrs[] = { &p->a, &p->b, &p->c };

    float* iter = mem;
    for (int i = 0; i < NUM_TENSORS; i++) {
        *(ptrs[i]) = iter;
        iter += sizes[i];
    }
    return mem;
}

int main(void) {
    size_t sizes[NUM_TENSORS] = {5, 3, 4};
    ToyParams p;
    float* mem = malloc_and_point(&p, sizes);

    // Fill via the typed handles
    for (int i = 0; i < 5; i++) p.a[i] = 1.0f + i;
    for (int i = 0; i < 3; i++) p.b[i] = 10.0f + i;
    for (int i = 0; i < 4; i++) p.c[i] = 100.0f + i;

    // The magic: dump the whole `mem` block as one flat array
    printf("Underlying flat buffer (12 floats):\n  ");
    for (int i = 0; i < 12; i++) printf("%.0f ", mem[i]);
    printf("\n");

    printf("Same bytes via typed pointers:\n");
    printf("  p.a (size 5) at offset %ld : ", (long)(p.a - mem));
    for (int i = 0; i < 5; i++) printf("%.0f ", p.a[i]);
    printf("\n");
    printf("  p.b (size 3) at offset %ld : ", (long)(p.b - mem));
    for (int i = 0; i < 3; i++) printf("%.0f ", p.b[i]);
    printf("\n");
    printf("  p.c (size 4) at offset %ld : ", (long)(p.c - mem));
    for (int i = 0; i < 4; i++) printf("%.0f ", p.c[i]);
    printf("\n");

    free(mem);
    return 0;
}


In [ ]:
!gcc -O2 -Wall -o course/ch08_build/unified_malloc course/ch08_build/unified_malloc.c && ./course/ch08_build/unified_malloc


You should see the three logical tensors `[1,2,3,4,5]`, `[10,11,12]`, `[100,101,102,103]` laid out **adjacent** in `mem`. The three named pointers (`p.a`, `p.b`, `p.c`) point at offsets `0`, `5`, `8` of the same block.

This is exactly how `params.wte`, `params.wpe`, `params.ln1w`, … all share `params_memory` in `train_gpt2.c`. The only difference is scale — 16 fields, ~124M floats.


## 11. Demo 2 — One AdamW Step, Verified Against PyTorch

Now the optimizer. We'll write `gpt2_update`'s inner loop on a tiny parameter array, and run `torch.optim.AdamW` on the same array, and confirm bit-for-bit match.


In [ ]:
%%writefile course/ch08_build/adamw_step.c
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

void adamw_step(float* params, float* grads, float* m, float* v, size_t n,
                float lr, float beta1, float beta2, float eps, float wd, int t) {
    for (size_t i = 0; i < n; i++) {
        float p = params[i];
        float g = grads[i];

        m[i] = beta1 * m[i] + (1.0f - beta1) * g;
        v[i] = beta2 * v[i] + (1.0f - beta2) * g * g;
        float m_hat = m[i] / (1.0f - powf(beta1, t));
        float v_hat = v[i] / (1.0f - powf(beta2, t));

        params[i] -= lr * (m_hat / (sqrtf(v_hat) + eps) + wd * p);
    }
}

static void* rd(const char* p, size_t n) {
    FILE* f = fopen(p, "rb"); if (!f){perror(p); exit(1);}
    void* b = malloc(n); size_t r = fread(b,1,n,f); (void)r; fclose(f); return b;
}

int main(int argc, char** argv) {
    if (argc != 8) { fprintf(stderr, "usage: n lr b1 b2 eps wd t\n"); return 1; }
    size_t n = (size_t) atoi(argv[1]);
    float lr=atof(argv[2]), b1=atof(argv[3]), b2=atof(argv[4]);
    float eps=atof(argv[5]), wd=atof(argv[6]); int t=atoi(argv[7]);

    float* params = (float*) rd("course/ch08_build/params.bin", n*sizeof(float));
    float* grads  = (float*) rd("course/ch08_build/grads.bin",  n*sizeof(float));
    float* m      = (float*) rd("course/ch08_build/m.bin",      n*sizeof(float));
    float* v      = (float*) rd("course/ch08_build/v.bin",      n*sizeof(float));

    adamw_step(params, grads, m, v, n, lr, b1, b2, eps, wd, t);

    FILE* f;
    f=fopen("course/ch08_build/params_after.bin","wb"); fwrite(params,4,n,f); fclose(f);
    f=fopen("course/ch08_build/m_after.bin",     "wb"); fwrite(m,     4,n,f); fclose(f);
    f=fopen("course/ch08_build/v_after.bin",     "wb"); fwrite(v,     4,n,f); fclose(f);
    free(params); free(grads); free(m); free(v); return 0;
}


In [ ]:
!gcc -O3 -Wall -o course/ch08_build/adamw_step course/ch08_build/adamw_step.c -lm


In [ ]:
# Run our C AdamW step and torch.optim.AdamW on identical inputs, compare
import numpy as np, torch, subprocess

torch.manual_seed(0)
n = 64
lr, beta1, beta2, eps, wd = 1e-3, 0.9, 0.999, 1e-8, 0.01
t = 1

# Initial state
params0 = torch.randn(n).float()
grads0  = torch.randn(n).float()
m0 = torch.zeros(n).float()
v0 = torch.zeros(n).float()

# Save inputs for the C side
params0.numpy().tofile("course/ch08_build/params.bin")
grads0.numpy().tofile("course/ch08_build/grads.bin")
m0.numpy().tofile("course/ch08_build/m.bin")
v0.numpy().tofile("course/ch08_build/v.bin")

# Run C
subprocess.run(["./course/ch08_build/adamw_step",
                str(n), str(lr), str(beta1), str(beta2), str(eps), str(wd), str(t)], check=True)
params_after_c = np.fromfile("course/ch08_build/params_after.bin", dtype=np.float32)
m_after_c      = np.fromfile("course/ch08_build/m_after.bin",      dtype=np.float32)
v_after_c      = np.fromfile("course/ch08_build/v_after.bin",      dtype=np.float32)

# Run torch.optim.AdamW on the same numbers
p_t = params0.clone().requires_grad_()
opt = torch.optim.AdamW([p_t], lr=lr, betas=(beta1, beta2), eps=eps, weight_decay=wd)
# inject a fake gradient
p_t.grad = grads0.clone()
opt.step()

print(f"params after step  diff: {np.max(np.abs(params_after_c - p_t.detach().numpy())):.2e}")
# pull AdamW's internal m, v out for comparison
state = opt.state[p_t]
print(f"m after step       diff: {np.max(np.abs(m_after_c - state['exp_avg'].numpy())):.2e}")
print(f"v after step       diff: {np.max(np.abs(v_after_c - state['exp_avg_sq'].numpy())):.2e}")


All three buffers should match `torch.optim.AdamW` to within float32 noise. **You just implemented a production optimizer in 18 lines of C** — and it's bit-equivalent to PyTorch's.


## 12. Verifying Loss Decrease Over Multiple Steps

Let's make the AdamW story concrete by chaining several steps and watching a quadratic loss decrease.


In [ ]:
# 50 AdamW steps on a 1-D quadratic loss L = 0.5 * (theta - target)^2
# (gradient is just (theta - target)). Confirms our C step works iteratively.
import numpy as np, torch, subprocess

n = 64
target = torch.randn(n)
theta_c = torch.randn(n).numpy().astype(np.float32)
theta_t = torch.from_numpy(theta_c).clone().requires_grad_()

m_c = np.zeros(n, dtype=np.float32); v_c = np.zeros(n, dtype=np.float32)
opt = torch.optim.AdamW([theta_t], lr=1e-1, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0)

losses_c, losses_t = [], []
for t in range(1, 51):
    # gradient of the quadratic
    grad = (theta_c - target.numpy().astype(np.float32))

    # save inputs
    theta_c.tofile("course/ch08_build/params.bin")
    grad.tofile("course/ch08_build/grads.bin")
    m_c.tofile("course/ch08_build/m.bin")
    v_c.tofile("course/ch08_build/v.bin")
    subprocess.run(["./course/ch08_build/adamw_step",
                    str(n), "0.1", "0.9", "0.999", "1e-8", "0.0", str(t)], check=True)
    theta_c = np.fromfile("course/ch08_build/params_after.bin", dtype=np.float32)
    m_c     = np.fromfile("course/ch08_build/m_after.bin",      dtype=np.float32)
    v_c     = np.fromfile("course/ch08_build/v_after.bin",      dtype=np.float32)
    losses_c.append(0.5 * float(((theta_c - target.numpy())**2).sum()))

    # PyTorch counterpart — measure loss AFTER the step (same as the C side)
    loss = 0.5 * ((theta_t - target)**2).sum()
    opt.zero_grad(); loss.backward(); opt.step()
    losses_t.append(0.5 * ((theta_t.detach() - target)**2).sum().item())

print(f"step  C loss        torch loss      diff")
for i in [0, 4, 9, 24, 49]:
    print(f"{i+1:4d}  {losses_c[i]:>12.6f}  {losses_t[i]:>12.6f}  {abs(losses_c[i]-losses_t[i]):.2e}")
print(f"\nfinal C loss   = {losses_c[-1]:.6f}")
print(f"final torch loss = {losses_t[-1]:.6f}")
print("✓ both losses go to ~0, both converge in lock-step." if losses_c[-1] < 1e-3 else "✗ check the AdamW step code")


Loss should go from ~`30+` to nearly zero over 50 steps, and the C and PyTorch losses should agree to ~`1e-6` per step. **AdamW works.** It's the same loop that runs every step of GPT-2 training, just on 64 parameters instead of 124M.


## 13. (Optional) Running the Real Thing — `test_gpt2`

If you have time and bandwidth, run the repo's official correctness test:

```bash
./dev/download_starter_pack.sh                 # ~520 MB — GPT-2 124M weights, tokenizer, debug state
make test_gpt2
./test_gpt2
```

`test_gpt2.c` does what we've been doing manually all chapter:

1. Load actual GPT-2 124M weights from `gpt2_124M.bin`.
2. Load PyTorch-generated debug tensors from `gpt2_124M_debug_state.bin` — including expected logits, expected loss, expected gradients of every parameter, and expected post-AdamW weights.
3. Run `gpt2_forward` and check logits/loss against the expected values.
4. Run `gpt2_backward` and check every gradient tensor.
5. Run 10 training iterations of AdamW and confirm the loss decreases to roughly the PyTorch reference.

If you've followed Chapters 2–8, **every single layer this test exercises is one you wrote yourself**. The test passing is your proof that the chapter contents are bit-equivalent to PyTorch's GPT-2 training.

The starter-pack download is optional and not needed to complete the course — Demos 1, 2, and 12 above already verify the milestone numerically.


## 14. TODO Exercise 1 — Implement `malloc_and_point`

Boilerplate provided. Fill in the loop that assigns each tensor's pointer to its slice of the big buffer.


In [ ]:
%%writefile course/ch08_build/exercise1.c
#include <stdio.h>
#include <stdlib.h>

#define NUM_TENSORS 3
typedef struct {
    float* a;
    float* b;
    float* c;
} ToyParams;

float* malloc_and_point(ToyParams* p, size_t* sizes) {
    size_t total = 0;
    for (int i = 0; i < NUM_TENSORS; i++) total += sizes[i];
    float* mem = (float*) malloc(total * sizeof(float));
    if (!mem) { perror("malloc"); exit(1); }

    float** ptrs[] = { &p->a, &p->b, &p->c };

    // TODO: walk through `mem`, assigning each *(ptrs[i]) to point at the right offset.
    // Hint: maintain a `float* iter = mem;` and advance it by `sizes[i]` each iteration.

    return mem;
}

int main(void) {
    size_t sizes[NUM_TENSORS] = {5, 3, 4};
    ToyParams p; float* mem = malloc_and_point(&p, sizes);

    for (int i = 0; i < 5; i++) p.a[i] = 1.0f + i;
    for (int i = 0; i < 3; i++) p.b[i] = 10.0f + i;
    for (int i = 0; i < 4; i++) p.c[i] = 100.0f + i;

    // Save the underlying buffer for the auto-grader
    FILE* f = fopen("course/ch08_build/mem_ex1.bin","wb"); fwrite(mem,4,12,f); fclose(f);
    free(mem); return 0;
}


In [ ]:
# Auto-grade Exercise 1
import numpy as np, subprocess
subprocess.run(["gcc","-O2","-Wall","-o","course/ch08_build/exercise1","course/ch08_build/exercise1.c"], check=True)
subprocess.run(["./course/ch08_build/exercise1"], check=True)
mem = np.fromfile("course/ch08_build/mem_ex1.bin", dtype=np.float32)
expected = np.array([1,2,3,4,5, 10,11,12, 100,101,102,103], dtype=np.float32)
err = np.max(np.abs(mem - expected))
print(f"diff: {err:.2e}")
print(f"mem: {mem.tolist()}")
print("PASS" if err == 0 else "FAIL — check your iter += sizes[i] step")


### Solution to Exercise 1

In [ ]:
%%writefile course/ch08_build/exercise1_sol.c
#include <stdio.h>
#include <stdlib.h>

#define NUM_TENSORS 3
typedef struct { float* a; float* b; float* c; } ToyParams;

float* malloc_and_point(ToyParams* p, size_t* sizes) {
    size_t total = 0;
    for (int i = 0; i < NUM_TENSORS; i++) total += sizes[i];
    float* mem = (float*) malloc(total * sizeof(float));
    if (!mem) { perror("malloc"); exit(1); }

    float** ptrs[] = { &p->a, &p->b, &p->c };

    float* iter = mem;
    for (int i = 0; i < NUM_TENSORS; i++) {
        *(ptrs[i]) = iter;
        iter += sizes[i];
    }
    return mem;
}

int main(void) {
    size_t sizes[NUM_TENSORS] = {5, 3, 4};
    ToyParams p; float* mem = malloc_and_point(&p, sizes);
    for (int i = 0; i < 5; i++) p.a[i] = 1.0f + i;
    for (int i = 0; i < 3; i++) p.b[i] = 10.0f + i;
    for (int i = 0; i < 4; i++) p.c[i] = 100.0f + i;
    FILE* f = fopen("course/ch08_build/mem_ex1.bin","wb"); fwrite(mem,4,12,f); fclose(f);
    free(mem); return 0;
}


In [ ]:
!gcc -O2 -Wall -o course/ch08_build/exercise1_sol course/ch08_build/exercise1_sol.c && ./course/ch08_build/exercise1_sol && echo ran


## 15. TODO Exercise 2 — Implement One AdamW Step

Boilerplate has the loop and reads `params`, `grads`, `m`, `v`. Fill in the **5-line update body** from the math in Section 8.


In [ ]:
%%writefile course/ch08_build/exercise2.c
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

void adamw_step(float* params, float* grads, float* m, float* v, size_t n,
                float lr, float beta1, float beta2, float eps, float wd, int t) {
    for (size_t i = 0; i < n; i++) {
        float p = params[i];
        float g = grads[i];

        // TODO 1: m[i] = beta1*m[i] + (1-beta1)*g
        // TODO 2: v[i] = beta2*v[i] + (1-beta2)*g*g
        // TODO 3: m_hat = m[i] / (1 - beta1^t)
        // TODO 4: v_hat = v[i] / (1 - beta2^t)
        // TODO 5: params[i] -= lr * ( m_hat / (sqrt(v_hat) + eps) + wd*p )

        (void)p; (void)g;
    }
}

static void* rd(const char* p, size_t n){FILE*f=fopen(p,"rb");void*b=malloc(n);size_t r=fread(b,1,n,f);(void)r;fclose(f);return b;}

int main(int argc, char** argv) {
    if (argc != 8) return 1;
    size_t n = (size_t) atoi(argv[1]);
    float lr=atof(argv[2]), b1=atof(argv[3]), b2=atof(argv[4]);
    float eps=atof(argv[5]), wd=atof(argv[6]); int t=atoi(argv[7]);
    float* params = (float*) rd("course/ch08_build/params.bin", n*sizeof(float));
    float* grads  = (float*) rd("course/ch08_build/grads.bin",  n*sizeof(float));
    float* m      = (float*) rd("course/ch08_build/m.bin",      n*sizeof(float));
    float* v      = (float*) rd("course/ch08_build/v.bin",      n*sizeof(float));
    adamw_step(params, grads, m, v, n, lr, b1, b2, eps, wd, t);
    FILE* f;
    f=fopen("course/ch08_build/params_ex2.bin","wb"); fwrite(params,4,n,f); fclose(f);
    free(params); free(grads); free(m); free(v); return 0;
}


In [ ]:
# Auto-grade Exercise 2
import numpy as np, torch, subprocess
torch.manual_seed(0); n = 64
params0 = torch.randn(n).float()
grads0  = torch.randn(n).float()
params0.numpy().tofile("course/ch08_build/params.bin")
grads0.numpy().tofile("course/ch08_build/grads.bin")
np.zeros(n, dtype=np.float32).tofile("course/ch08_build/m.bin")
np.zeros(n, dtype=np.float32).tofile("course/ch08_build/v.bin")
subprocess.run(["gcc","-O3","-Wall","-o","course/ch08_build/exercise2","course/ch08_build/exercise2.c","-lm"], check=True)
subprocess.run(["./course/ch08_build/exercise2", str(n),"1e-3","0.9","0.999","1e-8","0.01","1"], check=True)
params_c = np.fromfile("course/ch08_build/params_ex2.bin", dtype=np.float32)
p_t = params0.clone().requires_grad_(); p_t.grad = grads0.clone()
opt = torch.optim.AdamW([p_t], lr=1e-3, betas=(0.9,0.999), eps=1e-8, weight_decay=0.01)
opt.step()
err = np.max(np.abs(params_c - p_t.detach().numpy()))
print(f"params diff: {err:.2e}")
print("PASS" if err < 1e-6 else "FAIL — recheck the 5 AdamW lines")


### Solution to Exercise 2

In [ ]:
%%writefile course/ch08_build/exercise2_sol.c
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

void adamw_step(float* params, float* grads, float* m, float* v, size_t n,
                float lr, float beta1, float beta2, float eps, float wd, int t) {
    for (size_t i = 0; i < n; i++) {
        float p = params[i];
        float g = grads[i];
        m[i] = beta1*m[i] + (1.0f - beta1)*g;
        v[i] = beta2*v[i] + (1.0f - beta2)*g*g;
        float m_hat = m[i] / (1.0f - powf(beta1, t));
        float v_hat = v[i] / (1.0f - powf(beta2, t));
        params[i] -= lr * (m_hat / (sqrtf(v_hat) + eps) + wd*p);
    }
}

static void* rd(const char* p, size_t n){FILE*f=fopen(p,"rb");void*b=malloc(n);size_t r=fread(b,1,n,f);(void)r;fclose(f);return b;}

int main(int argc, char** argv) {
    if (argc != 8) return 1;
    size_t n = (size_t) atoi(argv[1]);
    float lr=atof(argv[2]), b1=atof(argv[3]), b2=atof(argv[4]);
    float eps=atof(argv[5]), wd=atof(argv[6]); int t=atoi(argv[7]);
    float* params = (float*) rd("course/ch08_build/params.bin", n*sizeof(float));
    float* grads  = (float*) rd("course/ch08_build/grads.bin",  n*sizeof(float));
    float* m      = (float*) rd("course/ch08_build/m.bin",      n*sizeof(float));
    float* v      = (float*) rd("course/ch08_build/v.bin",      n*sizeof(float));
    adamw_step(params, grads, m, v, n, lr, b1, b2, eps, wd, t);
    FILE* f;
    f=fopen("course/ch08_build/params_ex2.bin","wb"); fwrite(params,4,n,f); fclose(f);
    free(params); free(grads); free(m); free(v); return 0;
}


In [ ]:
!gcc -O3 -Wall -o course/ch08_build/exercise2_sol course/ch08_build/exercise2_sol.c -lm && echo ran


## Recap — End of Part I

You now know:

- A `(124M floats × 4 bytes)` model is held as **one giant `malloc`** with a 16-pointer typed-handle struct on top — `params_memory` for parameters, `grads_memory` for gradients, `m_memory` and `v_memory` for AdamW state. Four buffers, all the same size, all aligned at the same offsets.
- The **lazy-allocate-on-first-use** pattern for activations (size depends on `B, T`), gradients, and optimizer state.
- `gpt2_forward` is **encoder + 12×(LN→QKV→attn→proj→residual→LN→FFN→residual) + LN+head**, and the head reuses `params.wte` (weight tying).
- `gpt2_backward` runs the layers in reverse, all using `+=` so that residual paths and weight tying combine naturally.
- AdamW is **18 lines of C**: smoothed first/second moments, bias correction, decoupled weight decay. Bit-equivalent to `torch.optim.AdamW`.

Most importantly: **you've now read every line of `train_gpt2.c` that does math, and you understand all of it.** That's a milestone worth pausing on.

### What's next — Part II: GPU

**Chapter 9 — Hello, GPU.** We pivot. Everything you've built runs on a CPU. Now we'll meet the CUDA programming model: `__global__` kernels, `<<<grid, block>>>` launch syntax, `cudaMalloc` / `cudaMemcpy`, and the host↔device split. We'll port `gelu_forward` (the simplest layer in `llm.c`) to CUDA in 10 lines and watch it run on hundreds of threads in parallel.

If you don't have a GPU, you can still follow Part II conceptually — the code listings and explanations stand on their own, and Google Colab's free T4 GPUs run every CUDA cell in this course.
